In [1]:
import pandas as pd
import numpy as np

dictionary = pd.read_csv("data/data_dictionary.csv")

In [2]:
dictionary.columns

Index(['source_bundle', 'source_system_id', 'file_name', 'field_name',
       'logical_type', 'key_role', 'description', 'example_value'],
      dtype='str')

Essayer de comprendre la structure des données 

In [3]:
dictionary[dictionary["file_name"].str.contains(
    "clinical_observation",
    case=False,
    na=False
)]

,source_bundle,source_system_id,file_name,field_name,logical_type,key_role,description,example_value
12,SB02,SYS-CLN-OPS,clinical_observations.csv,observation_number,identifier,primary_or_composite_key,Observation number.,OBS-0000001
13,SB02,SYS-CLN-OPS,clinical_observations.csv,encounter_number,identifier,non_key,Source-local encounter identifier.,ENC-000001
14,SB02,SYS-CLN-OPS,clinical_observations.csv,patient_number,identifier,non_key,Source-local patient identifier used within th...,PAT-31162
15,SB02,SYS-CLN-OPS,clinical_observations.csv,observation_time,date_or_datetime,non_key,Observation time.,2018-02-27T09:27:00
16,SB02,SYS-CLN-OPS,clinical_observations.csv,observation_code,identifier,non_key,Observation code.,RPULSE
17,SB02,SYS-CLN-OPS,clinical_observations.csv,observation_name,categorical_or_text,non_key,Observation name.,Resting pulse
18,SB02,SYS-CLN-OPS,clinical_observations.csv,numeric_value,numeric,non_key,Numeric value.,64.0
19,SB02,SYS-CLN-OPS,clinical_observations.csv,text_value,categorical_or_text,non_key,Text value.,inconclusive
20,SB02,SYS-CLN-OPS,clinical_observations.csv,unit,categorical_or_text,non_key,Unit.,bpm
21,SB02,SYS-CLN-OPS,clinical_observations.csv,reference_range_text,categorical_or_text,non_key,Reference range text.,50–100


In [4]:
def audit_dataframe(df, name):
    print("=" * 60)
    print(name)
    print("=" * 60)

    print("Dimensions :", df.shape)

    print("\nTypes :")
    print(df.dtypes)

    print("\nValeurs manquantes :")
    print(df.isna().sum())

    print("\nDoublons complets :", df.duplicated().sum())

    print("\nNombre de valeurs uniques :")
    print(df.nunique())

    print()

In [5]:
recordings = pd.read_csv("data/health/cardiac_recordings.csv")
signal_samples = pd.read_csv("data/health/cardiac_signal_samples.csv")
encounters = pd.read_csv("data/health/clinical_encounters.csv")
notes = pd.read_csv("data/health/clinical_notes.csv")
observation = pd.read_csv("data/health/clinical_observations.csv")
clinical = pd.read_csv("data/health/clinical_procedures.csv")
patients = pd.read_csv("data/health/patient_administration.csv")

audit_dataframe(recordings, "cardiac_recordings")
audit_dataframe(signal_samples, "cardiac_signal_samples")
audit_dataframe(encounters, "clinical_encounters")
audit_dataframe(notes, "clinical_notes")
audit_dataframe(observation, "clinical_observation")
audit_dataframe(clinical, "clinical_procedures")
audit_dataframe(patients, "patient_administration")

cardiac_recordings
Dimensions : (44, 19)

Types :
recording_id                    str
patient_number                  str
encounter_number                str
recording_date                  str
recording_context               str
facility_license_number         str
reviewing_license_number        str
device_serial_number            str
device_model                    str
sampling_rate_hz              int64
sampling_interval_ms          int64
duration_seconds            float64
channel_count                 int64
sample_count                  int64
amplitude_unit                  str
automated_interpretation        str
recording_notes_code            str
device_quality              float64
archive_status                  str
dtype: object

Valeurs manquantes :
recording_id                0
patient_number              0
encounter_number            0
recording_date              0
recording_context           0
facility_license_number     0
reviewing_license_number    0
device_serial_number

In [6]:
datasets = {
    "cardiac_recordings": recordings,
    "cardiac_signal_samples": signal_samples,
    "clinical_encounters": encounters,
    "clinical_notes": notes,
    "clinical_observation": observation,
    "clinical_procedures": clinical,
    "patient_administration": patients
}

Plusieurs problemes commun a tout les fichiers : 
- la standardisation du format des dates
- verifier que date de sortie < date d'entrée 
- variables cat
- supprimer les doublons parfait
- gerer les valeurs manquantes

Autres : 
- meme archive_status pour tous (finalized) (dans cardiac_recordings)
- ...

## 1. Gerer les redondances

In [7]:
# je veux voir les lignes de doublons parfaits 
for name, df in datasets.items():
    duplicates = df[df.duplicated(keep=False)]
    
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    
    if duplicates.empty:
        print("No perfect duplicates found.")
    else:
        display(duplicates)


cardiac_recordings
No perfect duplicates found.

cardiac_signal_samples
No perfect duplicates found.

clinical_encounters


,encounter_number,patient_number,encounter_start,encounter_end,facility_license_number,encounter_type,referral_source,attending_license_number,service_unit,disposition,confidentiality_class,follow_up_date
732,ENC-000733,PAT-31066,2023-06-17T12:15:00,2023-06-17T13:59:00,FAC-VLN-101,general_examination,executive health office,LIC-VLN-MED-1003,specialist review,follow_up_planned,standard,2024-02-21
845,ENC-000846,PAT-31108,2022-05-04T12:15:00,2022-05-04T13:25:00,FAC-AVL-201,occupational_assessment,self,LIC-VLN-MED-1005,mobility laboratory,follow_up_planned,standard,2022-10-25
1067,ENC-000846,PAT-31108,2022-05-04T12:15:00,2022-05-04T13:25:00,FAC-AVL-201,occupational_assessment,self,LIC-VLN-MED-1005,mobility laboratory,follow_up_planned,standard,2022-10-25
1068,ENC-000733,PAT-31066,2023-06-17T12:15:00,2023-06-17T13:59:00,FAC-VLN-101,general_examination,executive health office,LIC-VLN-MED-1003,specialist review,follow_up_planned,standard,2024-02-21



clinical_notes
No perfect duplicates found.

clinical_observation


,observation_number,encounter_number,patient_number,observation_time,observation_code,observation_name,numeric_value,text_value,unit,reference_range_text,interpretation_code,device_reference,quality_code,assessor_license_number
3235,OBS-0003236,ENC-000245,PAT-31192,2021-09-01T11:32:00,MOBILITY,Mobility score,90.4,NaN,index,0–100,within_expected_range,DEV-006,acceptable,LIC-VLN-RES-1016
15194,OBS-0015195,ENC-000436,PAT-31157,2022-12-05T13:58:00,BINOCOORD,Binocular coordination,NaN,reduced,NaN,normal; reduced; atypical; not tested,recorded,NaN,acceptable,LIC-VLN-MED-1005
17734,OBS-0015195,ENC-000436,PAT-31157,2022-12-05T13:58:00,BINOCOORD,Binocular coordination,NaN,reduced,NaN,normal; reduced; atypical; not tested,recorded,NaN,acceptable,LIC-VLN-MED-1005
17735,OBS-0003236,ENC-000245,PAT-31192,2021-09-01T11:32:00,MOBILITY,Mobility score,90.4,NaN,index,0–100,within_expected_range,DEV-006,acceptable,LIC-VLN-RES-1016



clinical_procedures


,procedure_record_id,encounter_number,patient_number,procedure_code,procedure_description,scheduled_time,performed_time,performing_facility_license,performing_license_number,authorization_class,material_reference,procedure_status,outcome_code,next_review_date
262,PRC-000263,ENC-000334,PAT-31107,CARD-IMG,Advanced cardiac imaging,2021-10-31T09:35:00,2021-10-31T09:45:00,FAC-VLN-101,LIC-VLN-RES-1044,standard,NaN,completed,imaging_completed,2022-08-08
310,PRC-000263,ENC-000334,PAT-31107,CARD-IMG,Advanced cardiac imaging,2021-10-31T09:35:00,2021-10-31T09:45:00,FAC-VLN-101,LIC-VLN-RES-1044,standard,NaN,completed,imaging_completed,2022-08-08



patient_administration


,patient_number,family_name,given_name,date_of_birth,registered_sex,home_region,occupation_text,registration_date,primary_facility_license,record_status
173,PAT-31045,Garen,Olia,2000-08-01,female,Nordhaven Union,transport worker,2021-02-01,FAC-VLN-101,active
240,PAT-31045,Garen,Olia,2000-08-01,female,Nordhaven Union,transport worker,2021-02-01,FAC-VLN-101,active


In [8]:
# Suppression des doublons parfaits pour chaque dataset
for name, df in datasets.items():
    avant = df.shape[0]
    datasets[name] = df.drop_duplicates().copy() # On copie pour éviter les warnings
    apres = datasets[name].shape[0]
    if avant != apres:
        print(f"{name} : {avant - apres} doublon(s) supprimé(s).")

clinical_encounters : 2 doublon(s) supprimé(s).
clinical_observation : 2 doublon(s) supprimé(s).
clinical_procedures : 1 doublon(s) supprimé(s).
patient_administration : 1 doublon(s) supprimé(s).


## 2. Gerer les formats inconsistants

In [9]:
# Standardisation des dates 
colonnes_dates = dictionary[dictionary['logical_type'] == 'date_or_datetime']['field_name'].unique()

for name, df in datasets.items():
    cols_a_standartiser = [col for col in colonnes_dates if col in df.columns]
    
    for col in cols_a_standartiser:
        # On separe les dates avec slash et celles sans slash parce que le format de parsing est different
        mask_slash = df[col].astype(str).str.contains('/', na=False)
        
        dates_slash = pd.to_datetime(df.loc[mask_slash, col], format='mixed', dayfirst=True)
        dates_iso = pd.to_datetime(df.loc[~mask_slash, col], format='mixed')
        datasets[name][col] = pd.concat([dates_slash, dates_iso]).sort_index()
        
        print(f"Format mis à jour : '{col}' dans le dataset '{name}'")

# Affichage pour vérification
# display(datasets['clinical_encounters'][['encounter_start', 'encounter_end', 'follow_up_date']].head())

Format mis à jour : 'recording_date' dans le dataset 'cardiac_recordings'
Format mis à jour : 'encounter_start' dans le dataset 'clinical_encounters'
Format mis à jour : 'encounter_end' dans le dataset 'clinical_encounters'
Format mis à jour : 'follow_up_date' dans le dataset 'clinical_encounters'
Format mis à jour : 'document_date' dans le dataset 'clinical_notes'
Format mis à jour : 'observation_time' dans le dataset 'clinical_observation'
Format mis à jour : 'scheduled_time' dans le dataset 'clinical_procedures'
Format mis à jour : 'performed_time' dans le dataset 'clinical_procedures'
Format mis à jour : 'next_review_date' dans le dataset 'clinical_procedures'
Format mis à jour : 'date_of_birth' dans le dataset 'patient_administration'
Format mis à jour : 'registration_date' dans le dataset 'patient_administration'


In [10]:
# Vérification : date de fin antérieure à la date de début
anomalies_dates = datasets['clinical_encounters'][
    datasets['clinical_encounters']['encounter_end'] < datasets['clinical_encounters']['encounter_start']
]

print(f"Nombre d'incohérences de dates trouvées : {len(anomalies_dates)}")

Nombre d'incohérences de dates trouvées : 0


On va verifier maintenant si il n'y a pas de variables qui devraient etre numeriques mais qui sont des str et verifier pour quelles raisons (virgules a l'européennes au lieu du point, unités, faute de frappe) pour ensuite les parser et forcer le typage 

In [11]:
# On prends toutes les variables qui devraient etre des nombres 
colonnes_numeriques = dictionary[dictionary['logical_type'] == 'numeric']['field_name'].unique()

anomalie_detectee = False

for name, df in datasets.items():
    cols_a_verifier = [col for col in colonnes_numeriques if col in df.columns]
    
    for col in cols_a_verifier:
        if df[col].dtype == 'str':
            anomalie_detectee = True
            print(f"'{col}' dans le dataset '{name}' devrait être un nombre, mais est lu comme un str")

'numeric_value' dans le dataset 'clinical_observation' devrait être un nombre, mais est lu comme un str


In [12]:
# Simulation de la conversion pour repérer ce qui bloque
test_conversion = pd.to_numeric(datasets['clinical_observation']['numeric_value'], errors='coerce')

# Création d'un filtre pour isoler les valeurs non vides qui ont bloqué à la conversion
mask_aberrant = datasets['clinical_observation']['numeric_value'].notna() & test_conversion.isna()

print(f"Nombre total de valeurs mixtes (texte/chiffres) détectées : {mask_aberrant.sum()}")
display(datasets['clinical_observation'][mask_aberrant][['observation_name', 'numeric_value', 'unit']].head(10))

Nombre total de valeurs mixtes (texte/chiffres) détectées : 340


,observation_name,numeric_value,unit
29,Apical cardiac rate,"84,9",bpm
65,Cognitive score,82.2 index,index
167,Thoracic secondary peak index,"54,6",index
185,Resting pulse,"87,8",bpm
283,Estimated cardiac output,8.07 L/min,L/min
335,Peripheral oxygen saturation,96.0 %,%
378,Mobility score,"74,7",index
441,Lateral sway index,58.3 index,index
538,Resting pulse,"73,3",bpm
587,Balance score,69.8 index,index


In [13]:
import re

# Pour avoir un avant après (affichage)
datasets['clinical_observation']['valeur_brute'] = datasets['clinical_observation']['numeric_value']

def extraire_vrai_nombre(val):
    if pd.isna(val):
        return np.nan
    
    # Remplacement de la virgule européenne et nettoyage des espaces
    val_str = str(val).strip().replace(',', '.')
    
    # Capture : un signe moins (optionnel), des chiffres, un point décimal et des chiffres (optionnels)
    match = re.search(r'-?\d+(?:\.\d+)?', val_str)
    
    if match:
        return float(match.group(0))
    return np.nan

datasets['clinical_observation']['numeric_value'] = datasets['clinical_observation']['valeur_brute'].apply(extraire_vrai_nombre)

# (pour l'affichage)
masque_complexe = datasets['clinical_observation']['valeur_brute'].astype(str).str.contains(r'[a-zA-Z,]', regex=True, na=False)

print(f"Aperçu du nettoyage sur les {masque_complexe.sum()} valeurs complexes (avec unités ou virgules) :")
display(datasets['clinical_observation'].loc[masque_complexe, ['observation_name', 'valeur_brute', 'numeric_value']].head(10))

# 5. Forcer le typage final en float64 pour sécuriser la colonne
datasets['clinical_observation']['numeric_value'] = datasets['clinical_observation']['numeric_value'].astype(float)

# On supprime la colonne pour l'affichage
datasets['clinical_observation'] = datasets['clinical_observation'].drop(columns=['valeur_brute'])

print(f"\nTypage final de la colonne : {datasets['clinical_observation']['numeric_value'].dtype}")


Aperçu du nettoyage sur les 328 valeurs complexes (avec unités ou virgules) :


,observation_name,valeur_brute,numeric_value
29,Apical cardiac rate,"84,9",84.90
65,Cognitive score,82.2 index,82.20
167,Thoracic secondary peak index,"54,6",54.60
185,Resting pulse,"87,8",87.80
283,Estimated cardiac output,8.07 L/min,8.07
378,Mobility score,"74,7",74.70
441,Lateral sway index,58.3 index,58.30
538,Resting pulse,"73,3",73.30
587,Balance score,69.8 index,69.80
743,Peripheral pulse rate,"68,1",68.10



Typage final de la colonne : float64


## 3. Gerer les variables categorielles

On commence par un etat des lieux. On cherche les variables qui ont moins de 20 valeurs uniques ainsi que la liste exact des mots utilisés.

In [14]:
# Parcourir chaque dataset
for name, df in datasets.items():
    # Sélectionner uniquement les colonnes de type texte (object/string)
    cols_texte = df.select_dtypes(include=['object', 'string']).columns
    
    variables_categorielles_trouvees = False
    
    for col in cols_texte:
        # On ignore les colonnes d'identifiants ou de texte libre 
        nb_uniques = df[col].nunique()
        
        # On ignore aussi les dates (on les a déjà traitées)
        if 1 < nb_uniques <= 20 and 'date' not in col.lower() and 'time' not in col.lower():
            if not variables_categorielles_trouvees:
                print(f"\n{'='*50}\n{name}\n{'='*50}")
                variables_categorielles_trouvees = True
                
            valeurs_uniques = df[col].dropna().unique()
            print(f"➔ {col} ({nb_uniques} modalités) : {valeurs_uniques}")


cardiac_recordings
➔ recording_context (5 modalités) : <StringArray>
[              'rest',   'technical_repeat', 'clinical_follow_up',
      'post_exercise',  'stress_assessment']
Length: 5, dtype: str
➔ facility_license_number (3 modalités) : <StringArray>
['FAC-VLN-101', 'FAC-AVL-201', 'FAC-AVL-301']
Length: 3, dtype: str
➔ reviewing_license_number (8 modalités) : <StringArray>
['LIC-VLN-RES-1016', 'LIC-VLN-MED-1003', 'LIC-VLN-MED-1006',
 'LIC-VLN-MED-1001', 'LIC-VLN-MED-1002', 'LIC-VLN-MED-1005',
 'LIC-VLN-RES-1044', 'LIC-VLN-MED-1004']
Length: 8, dtype: str
➔ device_model (4 modalités) : <StringArray>
['Vantage Rhythm Monitor', 'HelixCard Duo', 'Asterion CV-3', 'Asterion CV-2']
Length: 4, dtype: str
➔ automated_interpretation (6 modalités) : <StringArray>
['complex', 'irregular', 'review_required', 'inconclusive', 'normal',
 'artefact']
Length: 6, dtype: str
➔ recording_notes_code (6 modalités) : <StringArray>
[    'rhythm_variability',     'repeat_recommended',                'n

In [15]:
# On retire les identifiants, licences, et les dates
# device_model et procedure_code sont bien des variables catégorielles mais elles sont déjà harmonisées donc on les retire
termes_exclus = ['id', 'license', 'number', 'date', 'time', 'reference']
colonnes_speciales_a_proteger = ['device_model', 'procedure_code']

for name, df in datasets.items():
    cols_texte = df.select_dtypes(include=['object', 'string']).columns
    
    for col in cols_texte:
        # On vérifie que la colonne n'est ni dans les termes exclus ni dans nos exceptions
        if not any(terme in col.lower() for terme in termes_exclus) and col not in colonnes_speciales_a_proteger:
            
            mask = df[col].notna()
            
            # Nettoyage des descriptions longues (garder la casse d'origine, juste nettoyer les espaces)
            if 'description' in col.lower() or col == 'text':
                df.loc[mask, col] = (
                    df[col][mask]
                    .astype(str)
                    .str.strip()
                    .str.replace(r'\s+', ' ', regex=True)
                )
            # Nettoyage complet (strip + lower) pour les variables catégorielles et les codes textuels
            else:
                df.loc[mask, col] = (
                    df[col][mask]
                    .astype(str)
                    .str.strip()
                    .str.lower()
                    .str.replace(r'\s+', ' ', regex=True)
                )
                print(f"Harmonisée : '{col}' dans '{name}'")

# Vérification rapide sur interpretation_code pour valider
print("\n Vérification (interpretation_code doit être en minuscules propres) :")
print(datasets['clinical_observation']['interpretation_code'].unique())

print("\n Vérification (procedure_code doit être resté intact) :")
print(datasets['clinical_procedures']['procedure_code'].unique())

Harmonisée : 'recording_context' dans 'cardiac_recordings'
Harmonisée : 'amplitude_unit' dans 'cardiac_recordings'
Harmonisée : 'automated_interpretation' dans 'cardiac_recordings'
Harmonisée : 'recording_notes_code' dans 'cardiac_recordings'
Harmonisée : 'archive_status' dans 'cardiac_recordings'
Harmonisée : 'encounter_type' dans 'clinical_encounters'
Harmonisée : 'referral_source' dans 'clinical_encounters'
Harmonisée : 'service_unit' dans 'clinical_encounters'
Harmonisée : 'disposition' dans 'clinical_encounters'
Harmonisée : 'document_type' dans 'clinical_notes'
Harmonisée : 'title' dans 'clinical_notes'
Harmonisée : 'observation_code' dans 'clinical_observation'
Harmonisée : 'observation_name' dans 'clinical_observation'
Harmonisée : 'text_value' dans 'clinical_observation'
Harmonisée : 'unit' dans 'clinical_observation'
Harmonisée : 'interpretation_code' dans 'clinical_observation'
Harmonisée : 'quality_code' dans 'clinical_observation'
Harmonisée : 'authorization_class' dans 'c

## 4. Gerer les donneés manquantes et aberrantes 

### 4.1 Données aberrantes

In [16]:
# On verifie si les dates sont coherentes entre elles 

# Croiser les observations avec les heures d'entrée et sortie du séjour
obs_croisees = pd.merge(
    datasets['clinical_observation'], 
    datasets['clinical_encounters'][['encounter_number', 'encounter_start', 'encounter_end']], 
    on='encounter_number', 
    how='inner',
    validate='m:1'  # Chaque observation correspond à un séjour unique
)

# Cibler les dates d'observation qui "débordent" du séjour
anomalies_temporelles = obs_croisees[
    (obs_croisees['observation_time'] < obs_croisees['encounter_start']) | 
    (obs_croisees['observation_time'] > obs_croisees['encounter_end'])
]

print(f"Observations saisies hors des limites du séjour : {len(anomalies_temporelles)}")

Observations saisies hors des limites du séjour : 0


on s'arrete la pour ne pas perdre des potentiels données utiles pour le but du TP 

### 4.2 Données manquantes

Les dates manquantes (follow_up_date et next_review_date) sont laissées tel quelle (NaT) pour les calculs et pour garder le type dateTime 

In [17]:
# La colonne superseded_document_id est entierement vide donc on la supprime
datasets['clinical_notes'] = datasets['clinical_notes'].drop(columns=['superseded_document_id'], errors='ignore')

In [18]:
# Les valeurs manquantes textes sont remplacées par un texte signifiant leur absence
datasets['clinical_observation']['device_reference'] = datasets['clinical_observation']['device_reference'].fillna('no_device')
datasets['clinical_procedures']['material_reference'] = datasets['clinical_procedures']['material_reference'].fillna('no_material')
datasets['clinical_encounters']['referral_source'] = datasets['clinical_encounters']['referral_source'].fillna('unspecified')

In [19]:
# On verifie si dans clinical_observation il y a toujours soit une numeric_value soit une text_value
# Sinon cela voudrait dire que le medecin a créé une fiche d'observation sans y inscrire le moindre résultat
obs_invalides = datasets['clinical_observation'][
    datasets['clinical_observation']['numeric_value'].isna() & 
    datasets['clinical_observation']['text_value'].isna()
]

print(f"Nombre d'observations sans texte ni chiffre : {len(obs_invalides)}")

Nombre d'observations sans texte ni chiffre : 0


Le resultat de lignes pour lesquelles il n'y a ni text_value ni numeric_value est nulle, on va donc remplacer les NaN de text_value en texte mais on garde les NaN pour les numeric_value car elle est non applicable par nature.

In [20]:
# S'il n'y a pas de texte, c'est que c'est une mesure numérique. On met donc "non_applicable"
datasets['clinical_observation']['text_value'] = datasets['clinical_observation']['text_value'].fillna('non_applicable')

On exporte tout dans des nouveau .csv pour garder le avant après

In [21]:
import os
dossier_source = './data/health' 
for name, df in datasets.items():
    nom_fichier = f"{name}_cleaned.csv"
    chemin_complet = os.path.join(dossier_source, nom_fichier)
    
    df.to_csv(chemin_complet, index=False)
    print(f"Exporté : {chemin_complet}")

Exporté : ./data/health/cardiac_recordings_cleaned.csv
Exporté : ./data/health/cardiac_signal_samples_cleaned.csv
Exporté : ./data/health/clinical_encounters_cleaned.csv
Exporté : ./data/health/clinical_notes_cleaned.csv
Exporté : ./data/health/clinical_observation_cleaned.csv
Exporté : ./data/health/clinical_procedures_cleaned.csv
Exporté : ./data/health/patient_administration_cleaned.csv
